#DQN

In [ ]:
!git clone https://github.com/icu-sepsis/icu-sepsis.git
%cd icu-sepsis/packages/
!pip install icu_sepsis/

Cloning into 'icu-sepsis'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 126 (delta 11), reused 5 (delta 3), pack-reused 92 (from 3)
Receiving objects: 100% (126/126), 3.76 MiB | 18.58 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/icu-sepsis/packages
Processing ./icu_sepsis
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for icu-sepsis: filename=icu_sepsis-2.0.1-py3-none-any.whl size=1208209 sha256=3c970a03b6392ad8929785305bb86ab04c2d860a7e254ae78d516cf4fbed0b04
  Stored in directory: /tmp/pip-ephem-wheel-cache-dru5nq6q/wheels/76/e6/a8/b6a8ba1780374b3600fb5184ed0a6ebc1330ff306538654266
Successfully built icu-sepsis


In [ ]:
import gymnasium as gym
import icu_sepsis

import numpy as np
import matplotlib
matplotlib.rcParams['agg.path.chunksize'] = 10000
import matplotlib.pyplot as plt
import os
import re
import glob
import random
from itertools import product
from pathlib import Path

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from gymnasium.vector import SyncVectorEnv

env = gym.make('Sepsis/ICU-Sepsis-v2')

state, info = env.reset()
print('Initial state:', state)
print('Extra info:', info)

next_state, reward, terminated, truncated, info = env.step(0)
print('\nTaking action 0:')
print('Next state:', next_state)
print('Reward:', reward)
print('Terminated:', terminated)
print('Truncated:', truncated)

print("Actions and State Space:")

print("Action space:", env.action_space)
print("State space:", env.observation_space)

Initial state: 317
Extra info: {'admissible_actions': [5, 10, 12, 15, 20], 'state_vector': array([-0.06428571,  0.34285714, -0.41176429,  0.47857143, -0.36184041,
       -0.06923292, -1.77363625,  0.69286201, -0.65947867, -0.44250746,
       -0.26254502,  0.33618265,  0.30801809,  0.24511727, -0.1161215 ,
       -0.02761093, -0.38035708, -0.17151222, -0.18541096, -0.09209938,
       -0.50079036,  0.13579159,  0.6025935 , -0.34943571, -0.34589266,
        0.50252722, -0.33549439,  0.37154391,  0.82892691,  0.81585198,
       -0.17917188,  0.44818029,  0.73517404,  0.95602272, -0.46129536,
       -0.21578368,  0.02364799, -0.58863746, -0.82986552, -0.59084129,
       -0.57710664, -0.288773  , -0.42938843,  0.36024281,  1.01333337,
        0.58052073,  0.49873189]), 'sofa_score': np.float64(7.916955017301038)}

Taking action 0:
Next state: 317
Reward: 0.0
Terminated: False
Truncated: False
Actions and State Space:
Action space: Discrete(25)
State space: Discrete(716)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ───────────────────────── helpers ───────────────────────────────
def one_hot(idx: torch.LongTensor, size: int, device=None):
    return F.one_hot(idx, num_classes=size).to(torch.float32).to(device)


# ─────────────────────── replay buffer ───────────────────────────
class ReplayBuffer:
    """Uniform replay with contiguous numpy arrays (faster than list)."""
    def __init__(self, capacity: int):
        self.cap = capacity
        self.pos = 0
        self.full = False
        self.s  = np.empty(capacity, dtype=np.int32)
        self.a  = np.empty(capacity, dtype=np.int16)
        self.r  = np.empty(capacity, dtype=np.float32)
        self.s2 = np.empty(capacity, dtype=np.int32)
        self.d  = np.empty(capacity, dtype=np.bool_)

    def add(self, s, a, r, s2, done):
        p = self.pos
        self.s [p] = s
        self.a [p] = a
        self.r [p] = r
        self.s2[p] = s2
        self.d [p] = done
        self.pos = (p + 1) % self.cap
        self.full |= self.pos == 0

    def sample(self, batch: int):
        N = self.cap if self.full else self.pos
        idx = np.random.randint(0, N, size=batch)
        return dict(
            s  = self.s [idx],
            a  = self.a [idx],
            r  = self.r [idx],
            s2 = self.s2[idx],
            d  = self.d [idx].astype(np.float32)
        )

    def __len__(self): return self.cap if self.full else self.pos


# ───────────────────────── network ───────────────────────────────
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
        self.apply(lambda m: nn.init.kaiming_uniform_(m.weight)
                   if isinstance(m, nn.Linear) else None)

    def forward(self, x): return self.net(x)


# ─────────────────────── vectorised DQN ──────────────────────────
class VectorDQN:
    def __init__(self,
                 env_id           ='Sepsis/ICU-Sepsis-v2',
                 num_envs         = 8,
                 lr               = 5e-4,
                 gamma            = 0.99,
                 buffer_size      = 200_000,
                 batch_size       = 128,
                 eps_start        = 1.0,
                 eps_end          = 0.01,
                 target_update    = 10_000,
                 learning_starts  = 10_000,
                 train_freq       = 4,
                 device='cpu'):

        # vector env --------------------------------------------------
        self.num_envs = num_envs
        def make(): return lambda: gym.make(env_id)
        self.envs  = SyncVectorEnv([make() for _ in range(num_envs)])
        obs, _     = self.envs.reset()
        self.S     = int(self.envs.single_observation_space.n)
        self.A     = int(self.envs.single_action_space.n)
        self.device = torch.device(device)

        # networks & optimiser ---------------------------------------
        self.q_net     = QNet(self.S, self.A).to(self.device)
        self.target_net= QNet(self.S, self.A).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.opt       = torch.optim.Adam(self.q_net.parameters(), lr=lr)

        # hyper-params & trackers ------------------------------------
        self.gamma = gamma
        self.eps_start, self.eps_end = eps_start, eps_end
        self.eps = eps_start
        self.target_update    = target_update
        self.learning_starts  = learning_starts
        self.train_freq_steps = train_freq * num_envs  # global step interval
        self.batch_size       = batch_size

        # replay & statistics ----------------------------------------
        self.replay = ReplayBuffer(buffer_size)
        self.obs    = obs
        self.global_step = 0
        self.learn_step  = 0
        self.global_epi  = 0
        self.max_episodes= None                     # set externally

        self.ret_running = np.zeros(num_envs, dtype=np.float32)
        self.len_running = np.zeros(num_envs, dtype=np.int32)
        self.ret_buf, self.len_buf = [], []

    # ε-greedy ------------------------------------------------------
    def _decay_eps(self):
        frac  = min(1.0, self.global_epi / (0.25 * self.max_episodes))
        self.eps = self.eps_start + frac * (self.eps_end - self.eps_start)

    def select_actions(self, obs):
        if random.random() < self.eps:
            return np.random.randint(self.A, size=self.num_envs)
        with torch.no_grad():
            oh = one_hot(torch.as_tensor(obs, dtype=torch.long,
                                         device=self.device),
                         self.S, self.device)
            return torch.argmax(self.q_net(oh), 1).cpu().numpy()

    # one vector-env step ------------------------------------------
    def step_envs(self):
        acts = self.select_actions(self.obs)
        nxt, rew, term, trunc, infos = self.envs.step(acts)
        done = term | trunc

        # track episode returns / lengths
        self.ret_running += rew
        self.len_running += 1
        finished = np.where(done)[0]
        if finished.size:
            self.ret_buf.extend(self.ret_running[finished])
            self.len_buf.extend(self.len_running[finished])
            self.ret_running[finished] = 0.0
            self.len_running[finished] = 0
            self.global_epi += finished.size

        # store transitions
        for i in range(self.num_envs):
            self.replay.add(self.obs[i], acts[i], rew[i], nxt[i], done[i])

        self.obs = nxt
        self.global_step += self.num_envs
        self._decay_eps()

        # maybe learn
        if (self.global_step > self.learning_starts and
            self.global_step % self.train_freq_steps == 0 and
            len(self.replay) >= self.batch_size):
            self.learn()

        return rew.mean(), done.mean()

    # SGD update ----------------------------------------------------
    def learn(self):
        batch = self.replay.sample(self.batch_size)

        s  = torch.as_tensor(batch['s'],  dtype=torch.long,   device=self.device)
        a  = torch.as_tensor(batch['a'],  dtype=torch.long,   device=self.device).unsqueeze(1)
        r  = torch.as_tensor(batch['r'],  dtype=torch.float32,device=self.device).unsqueeze(1)
        s2 = torch.as_tensor(batch['s2'], dtype=torch.long,   device=self.device)
        d  = torch.as_tensor(batch['d'],  dtype=torch.float32,device=self.device).unsqueeze(1)

        q_s = self.q_net(one_hot(s,  self.S, self.device)).gather(1, a)
        with torch.no_grad():
            q_next = self.target_net(one_hot(s2,self.S,self.device)
                       ).max(1, keepdim=True)[0]
            tgt = r + self.gamma * q_next * (1.0 - d)

        loss = F.mse_loss(q_s, tgt)

        self.opt.zero_grad(); loss.backward(); self.opt.step()

        self.learn_step += 1
        if self.learn_step % self.target_update == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

In [ ]:
# ------------------------------------------------------------------

base_dir = "/content/drive/MyDrive/COMP_579_Final_Project/DQN"
os.makedirs(base_dir, exist_ok=True)

# grid -----------------------------------------------------------------
seeds           = range(10)
learning_rates  = [1e-3, 5e-4]
gammas          = [0.99, 0.995]

episodes_total      = 300_000                   # completed eps
checkpoint_every    = 100_000
batch_size          = 128
num_envs            = 8

# ----------------------------------------------------------------------
for lr, gamma in product(learning_rates, gammas):
    tag      = f"VecDQN_lr{lr}_bs{batch_size}_gamma{gamma}"
    ckpt_dir = os.path.join(base_dir, tag)

    for seed in seeds:
        # reproducibility ---------------------------------------------
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

        # ----------------------------------------------------------------
        f_ret  = os.path.join(ckpt_dir, f"{tag}_seed{seed}_returns.npy")
        f_len  = os.path.join(ckpt_dir, f"{tag}_seed{seed}_lengths.npy")
        f_inad = os.path.join(ckpt_dir, f"{tag}_seed{seed}_inad.npy")

        returns = list(np.load(f_ret))  if os.path.exists(f_ret)  else []
        lengths = list(np.load(f_len))  if os.path.exists(f_len)  else []
        inads   = list(np.load(f_inad)) if os.path.exists(f_inad) else []

        # locate latest checkpoint ------------------------------------
        last_ckpt = 0
        if os.path.isdir(ckpt_dir):
            pat = re.compile(fr"{re.escape(tag)}_seed{seed}_epi(\d+)_q\.pth")
            for fn in os.listdir(ckpt_dir):
                m = pat.match(fn)
                if m:
                    last_ckpt = max(last_ckpt, int(m.group(1)))

        # instantiate agent -------------------------------------------
        agent = VectorDQN(num_envs        = num_envs,
                          lr              = lr,
                          gamma           = gamma,
                          buffer_size     = 200_000,
                          batch_size      = batch_size,
                          device          = "cpu")
        agent.max_episodes = episodes_total

        # restore weights if we have a checkpoint ---------------------
        if last_ckpt:
            agent.q_net.load_state_dict(
                torch.load(os.path.join(
                    ckpt_dir, f"{tag}_seed{seed}_epi{last_ckpt}_q.pth"),
                    map_location=agent.device))
            agent.target_net.load_state_dict(
                torch.load(os.path.join(
                    ckpt_dir, f"{tag}_seed{seed}_epi{last_ckpt}_target.pth"),
                    map_location=agent.device))
            agent.global_epi = last_ckpt    # resume ε schedule

        # -------------------------------------------------------------
        pbar = tqdm(total=episodes_total-agent.global_epi,
                    desc=f"{tag} | seed{seed}", unit="ep")

        while agent.global_epi < episodes_total:
            r_mean, d_mean = agent.step_envs()

            # flush newly finished episodes ---------------------------
            while len(returns) < len(agent.ret_buf):
                idx = len(returns)
                returns.append(float(agent.ret_buf[idx]))
                lengths.append(int  (agent.len_buf[idx]))
                inads.append(0.0)                  # inadmissible not tracked
                pbar.update(1)

            # checkpoint ---------------------------------------------
            if (agent.global_epi and
                agent.global_epi % checkpoint_every == 0):
                os.makedirs(ckpt_dir, exist_ok=True)
                torch.save(agent.q_net.state_dict(),
                           os.path.join(ckpt_dir,
                           f"{tag}_seed{seed}_epi{agent.global_epi}_q.pth"))
                torch.save(agent.target_net.state_dict(),
                           os.path.join(ckpt_dir,
                           f"{tag}_seed{seed}_epi{agent.global_epi}_target.pth"))
                np.save(f_ret,  np.array(returns))
                np.save(f_len,  np.array(lengths))
                np.save(f_inad, np.array(inads))

        pbar.close()

        # final save --------------------------------------------------
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save(agent.q_net.state_dict(),
                   os.path.join(ckpt_dir,
                   f"{tag}_seed{seed}_final_q.pth"))
        torch.save(agent.target_net.state_dict(),
                   os.path.join(ckpt_dir,
                   f"{tag}_seed{seed}_final_target.pth"))
        np.save(f_ret,  np.array(returns))
        np.save(f_len,  np.array(lengths))
        np.save(f_inad, np.array(inads))